In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, datasets, models
from torch.utils.data.dataloader import DataLoader
from torch.utils.tensorboard import SummaryWriter
from sklearn import preprocessing
import numpy as np
import copy
import os
import time

# PART 0: LARS Optimizer Implementation
class LARS(torch.optim.Optimizer):
    def __init__(self, params, lr, momentum=0.9, weight_decay=1e-4, trust_coefficient=0.001, eps=1e-8):
        defaults = dict(lr=lr, momentum=momentum, weight_decay=weight_decay,
                        trust_coefficient=trust_coefficient, eps=eps)
        super(LARS, self).__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            weight_decay = group['weight_decay']
            momentum = group['momentum']
            trust_coefficient = group['trust_coefficient']
            eps = group['eps']

            for p in group['params']:
                if p.grad is None:
                    continue
                d_p = p.grad

                p_norm = torch.norm(p.data)
                g_norm = torch.norm(d_p.data)

                if p_norm > 0 and g_norm > 0:
                    lars_lr = trust_coefficient * p_norm / (g_norm + p_norm * weight_decay + eps)
                    d_p = d_p.add(p, alpha=weight_decay)
                    d_p *= lars_lr

                param_state = self.state[p]
                if 'momentum_buffer' not in param_state:
                    buf = param_state['momentum_buffer'] = torch.clone(d_p).detach()
                else:
                    buf = param_state['momentum_buffer']
                    buf.mul_(momentum).add_(d_p)

                p.add_(buf, alpha=-group['lr'])
        return loss

# PART 1: Data Augmentation
class GaussianBlur(object):
    def __init__(self, kernel_size):
        radius = kernel_size // 2
        kernel_size = radius * 2 + 1
        self.blur_h = nn.Conv2d(3, 3, kernel_size=(kernel_size, 1), stride=1, padding=0, bias=False, groups=3)
        self.blur_v = nn.Conv2d(3, 3, kernel_size=(1, kernel_size), stride=1, padding=0, bias=False, groups=3)
        self.k, self.r = kernel_size, radius
        self.blur = nn.Sequential(nn.ReflectionPad2d(radius), self.blur_h, self.blur_v)
        self.pil_to_tensor = transforms.ToTensor()
        self.tensor_to_pil = transforms.ToPILImage()

    def __call__(self, img):
        img = self.pil_to_tensor(img).unsqueeze(0)
        sigma = np.random.uniform(0.1, 2.0)
        x = np.arange(-self.r, self.r + 1)
        x = np.exp(-np.power(x, 2) / (2 * sigma * sigma))
        x = x / x.sum()
        x = torch.from_numpy(x).view(1, -1).repeat(3, 1).float()
        self.blur_h.weight.data.copy_(x.view(3, 1, self.k, 1))
        self.blur_v.weight.data.copy_(x.view(3, 1, 1, self.k))
        with torch.no_grad():
            img = self.blur(img).squeeze()
        return self.tensor_to_pil(img)

class MultiViewDataInjector(object):
    def __init__(self, transform_list):
        self.transforms = transform_list
    def __call__(self, sample):
        return [t(sample) for t in self.transforms]

def get_simclr_data_transforms(input_shape, s=1):
    size = input_shape[0]
    color_jitter = transforms.ColorJitter(0.8 * s, 0.8 * s, 0.8 * s, 0.2 * s)
    return transforms.Compose([
        transforms.RandomResizedCrop(size=size),
        transforms.RandomHorizontalFlip(),
        transforms.RandomApply([color_jitter], p=0.8),
        transforms.RandomGrayscale(p=0.2),
        GaussianBlur(kernel_size=int(0.1 * size)),
        transforms.ToTensor()
    ])

# PART 2: Model Architectures
class MLPHead(nn.Module):
    def __init__(self, in_channels, mlp_hidden_size, projection_size):
        super(MLPHead, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(in_channels, mlp_hidden_size),
            nn.BatchNorm1d(mlp_hidden_size),
            nn.ReLU(inplace=True),
            nn.Linear(mlp_hidden_size,projection_size)
        )
    def forward(self, x):
        return self.net(x)



class ResNetEncoder(torch.nn.Module):
    def __init__(self, name, projection_head):
        super(ResNetEncoder, self).__init__()
        resnet = models.resnet50(weights=None) if name == 'resnet50' else models.resnet18(weights=None)
        self.encoder = torch.nn.Sequential(*list(resnet.children())[:-1])
        self.projection = MLPHead(in_channels=resnet.fc.in_features, **projection_head)

    def forward(self, x):
        h = self.encoder(x)
        h = h.view(h.shape[0], -1)
        return self.projection(h)

# PART 3: Multi-Target Trainer
class MultiTargetBYOLTrainer:
    def __init__(self, online_net, predictor, targets, optimizer, device, **params):
        self.online_net = online_net
        self.predictor = predictor
        self.targets = targets
        self.optimizer = optimizer
        self.device = device
        self.max_epochs, self.m = params['max_epochs'], params['m']
        self.batch_size, self.num_workers = params['batch_size'], params['num_workers']
        self.writer = SummaryWriter()

    def _update_targets(self):

        for target in self.targets:
            for p_q, p_k in zip(self.online_net.parameters(), target.parameters()):
                p_k.data = p_k.data * self.m + p_q.data * (1. - self.m)

    def regression_loss(self, x, y):
        x, y = F.normalize(x, dim=1), F.normalize(y, dim=1)
        return 2 - 2 * (x * y).sum(dim=-1)

    def train(self, train_dataset):
        loader = DataLoader(train_dataset, batch_size=self.batch_size, num_workers=self.num_workers, drop_last=True, shuffle=True)
        for epoch in range(self.max_epochs):
            for i, (views, _) in enumerate(loader):
                # views[0] is online, views[1:] are target views
                views = [v.to(self.device) for v in views]
                v_online = views[0]
                v_targets = views[1:]

                # Online forward
                p_online = self.predictor(self.online_net(v_online))

                # Multi-Target forward (Stop Gradient)
                total_loss = 0

                for idx, target_view in enumerate(v_targets):
                  with torch.no_grad():
                    z_target = self.targets[idx](target_view)
                        # Accumulate similarity loss across all branches
                  total_loss += self.regression_loss(p_online, z_target).mean()

                total_loss /= len(v_targets)

                self.optimizer.zero_grad()
                total_loss.backward()
                self.optimizer.step()

                self._update_targets()

                if i % 10 == 0:
                    print(f"Epoch {epoch}, Iter {i}, Multi-Target Loss (LARS): {total_loss.item():.4f}")

        save_path = os.path.join(self.writer.log_dir, 'multi_target_model_lars.pth')
        torch.save({'online_state': self.online_net.state_dict()}, save_path)
        print(f"Weights saved to {save_path}")
        return save_path

# PART 4: Linear Evaluation
def run_linear_evaluation(encoder, train_loader, test_loader, device):
    print("\n--- Linear Evaluation (Frozen Encoder) ---")
    encoder.eval()

    def extract_features(model, loader):
        x_out, y_out = [], []
        for img, lbl in loader:
            with torch.no_grad():
                img = img.to(device)
                f = model(img).view(img.size(0), -1).cpu()
                x_out.append(f); y_out.extend(lbl.numpy())
        return torch.cat(x_out, dim=0), torch.tensor(y_out)

    x_train, y_train = extract_features(encoder, train_loader)
    x_test, y_test = extract_features(encoder, test_loader)

    scaler = preprocessing.StandardScaler()
    x_train_scaled = torch.from_numpy(scaler.fit_transform(x_train)).float()
    x_test_scaled = torch.from_numpy(scaler.transform(x_test)).float()

    loader = DataLoader(torch.utils.data.TensorDataset(x_train_scaled, y_train), batch_size=256, shuffle=True)
    classifier = torch.nn.Linear(x_train_scaled.shape[1], 10).to(device)
    optimizer = torch.optim.Adam(classifier.parameters(), lr=3e-4)

    for epoch in range(100):
        for x_b, y_b in loader:
            optimizer.zero_grad()
            F.cross_entropy(classifier(x_b.to(device)), y_b.to(device)).backward()
            optimizer.step()

        if epoch % 10 == 0:
            with torch.no_grad():
                preds = torch.argmax(classifier(x_test_scaled.to(device)), dim=1)
                acc = (preds == y_test.to(device)).float().mean()
                print(f"Epoch {epoch} | Ensemble Accuracy: {acc.item()*100:.2f}%")

# PART 5: Execution
if __name__ == '__main__':
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    num_targets = 3

    # 1. Initialize Online Network (ResNet-50)
    online_net = ResNetEncoder('resnet50', {'mlp_hidden_size': 4096, 'projection_size': 256}).to(device)
    predictor = MLPHead(256, 4096, 256).to(device)

    # 2. Initialize Target Networks (EMA branches)
    targets = nn.ModuleList([
        copy.deepcopy(online_net) for _ in range(num_targets)
    ])
    for t in targets:
        for p in t.parameters(): p.requires_grad = False

    # 3. Configure LARS Optimizer with lr=0.02
    optimizer = LARS(
        list(online_net.parameters()) + list(predictor.parameters()),
        lr=0.02,
        momentum=0.9,
        weight_decay=1e-6
    )

    # 4. Data Preparation (N+1 views: 1 for online, N for targets)
    transform = get_simclr_data_transforms((96, 96, 3))
    train_ds = datasets.STL10('./data', split='train', download=True,
                               transform=MultiViewDataInjector([transform] * (num_targets + 1)))

    # 5. Training
    trainer = MultiTargetBYOLTrainer(online_net, predictor, targets, optimizer, device,
                                     max_epochs=500, m=0.996, batch_size=256, num_workers=2)
    trainer.train(train_ds)

    # 6. Final Evaluation
    eval_t = transforms.Compose([transforms.ToTensor()])
    tr_ldr = DataLoader(datasets.STL10('./data', split='train', transform=eval_t), batch_size=256)
    te_ldr = DataLoader(datasets.STL10('./data', split='test', transform=eval_t), batch_size=256)

    run_linear_evaluation(online_net.encoder, tr_ldr, te_ldr, device)

2026-03-19 05:40:15.988447: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773898816.185817      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773898816.238791      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773898816.709985      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773898816.710018      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773898816.710021      55 computation_placer.cc:177] computation placer alr

Epoch 0, Iter 0, Multi-Target Loss (LARS): 2.0017
Epoch 0, Iter 10, Multi-Target Loss (LARS): 1.8844
Epoch 1, Iter 0, Multi-Target Loss (LARS): 1.7184
Epoch 1, Iter 10, Multi-Target Loss (LARS): 1.5437
Epoch 2, Iter 0, Multi-Target Loss (LARS): 1.3760
Epoch 2, Iter 10, Multi-Target Loss (LARS): 1.2623
Epoch 3, Iter 0, Multi-Target Loss (LARS): 1.1524
Epoch 3, Iter 10, Multi-Target Loss (LARS): 1.0801
Epoch 4, Iter 0, Multi-Target Loss (LARS): 1.0105
Epoch 4, Iter 10, Multi-Target Loss (LARS): 0.9602
Epoch 5, Iter 0, Multi-Target Loss (LARS): 0.9466
Epoch 5, Iter 10, Multi-Target Loss (LARS): 0.8947
Epoch 6, Iter 0, Multi-Target Loss (LARS): 0.8622
Epoch 6, Iter 10, Multi-Target Loss (LARS): 0.8416
Epoch 7, Iter 0, Multi-Target Loss (LARS): 0.8306
Epoch 7, Iter 10, Multi-Target Loss (LARS): 0.8352
Epoch 8, Iter 0, Multi-Target Loss (LARS): 0.7936
Epoch 8, Iter 10, Multi-Target Loss (LARS): 0.8119
Epoch 9, Iter 0, Multi-Target Loss (LARS): 0.7711
Epoch 9, Iter 10, Multi-Target Loss (LARS